## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import colorcet as cc
import yaml
from tqdm import tqdm
from statsmodels.distributions.copula.api import GumbelCopula

import gumbel_copula_2dRP as rp
import RP_plotting as rp_plot

# set up plot preferences
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['font.size'] = 8

### Constants

In [ ]:
def load_config(config_path):
    '''Loads the configuration from a YAML file.'''
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

config = load_config('config/config.yaml')

In [ ]:
# Config parameters
THRESH = config['THRESH']
REGIONS = config['REGIONS']
# MODEL = config['MODEL']
MODEL = 'obs'
T = config['T']
# T = 50
n = config['n']

# Other parameters
EXPORTS = False
cmap = cc.cm['kbc_r']
u_vals = np.linspace(0.8, 0.999, n)
v_vals = np.linspace(0.8, 0.999, n)

### Data

In [ ]:
# Drought summary data
if MODEL == 'obs':
    # Obs
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/Drought_Properties_jd.csv',
        parse_dates=['start', 'end', 'previous_end'],
        index_col='StaID'
        )
    
elif MODEL == 'nwm':
    # NWM
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/nwm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']
    
elif MODEL == 'usgs':
    # USGS
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/lstm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']

else:
    # CBRFC
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/cbrfc_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']

# List of study gages with HCDN clusters
gages = pd.read_csv(
    '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/CRB Gages/NWM_v3_CRB_with_HCDN_cluster.csv',
    index_col='USGS_ID'
    )[['Lat', 'Lon', 'region']]

# Regions
regions = gages['region'].unique().tolist()
region_names = [
    'Southwest',
    'California and Interior West',
    'Rocky Mountains'
]
# print(regions[REGION])

## Fit the Marginals and Copula Function

In [ ]:
out_duration = []   # the most likely duration values
out_severity = []   # the most likely severity values


for i in tqdm(range(len(gages))):

    # Filter to region
    data = drought_data[drought_data['threshold'] == THRESH]
    data = data[data.index == gages.index[i]]
    data = data[['duration', 'severity']]
    data.dropna(inplace=True)
    
    if len(data) == 0:
        print(f'No data for gage {gages.index[i]}')
        out_duration.append([np.nan] * len(T))
        out_severity.append([np.nan] * len(T))

    else:

        # Fit EVs
        duration_rv, duration_params, duration_aic = rp.best_fit_rv(data['duration'], ['gamma', 'weibull_min', 'expon'], print=False)
        severity_rv, severity_params, severity_aic = rp.best_fit_rv(data['severity'], ['gamma', 'weibull_min', 'expon'], print=False)

        duration_rv = duration_rv(*duration_params)
        severity_rv = severity_rv(*severity_params)
        
        # fit theta
        copula = GumbelCopula()
        theta = copula.fit_corr_param(
            np.vstack((data['duration'], data['severity'])).transpose()
            )
        
        all_duration = []
        all_severity = []
        all_likelihood = []
        max_duration = []       # duration value with max likelihood
        max_severity = []       # severity value with max likelihood
        
        for t in T:
        
            # compute the iso-line
            V = rp.iso_rp_OR(u_vals, t, theta)
            
            # transform to duration and severity values
            duration_val = duration_rv.ppf(u_vals)
            severity_val = severity_rv.ppf(V)
            all_duration.append(duration_val)
            all_severity.append(severity_val)

            # compute the likelihood along iso-RP
            likelihood = rp.joint_density_OR(u_vals, V, theta, duration_rv, severity_rv)
            # Normalize likelihood to [0, 1], ignoring NaN values
            min_val = np.nanmin(likelihood)
            max_val = np.nanmax(likelihood)
            if max_val != min_val:
                likelihood_normalized = (likelihood - min_val) / (max_val - min_val)
            else:
                likelihood_normalized = np.zeros_like(likelihood)
            
            all_likelihood.append(likelihood_normalized)
            
            # get the duration and severity values at the max likelihood of the iso-line
            ind_max = np.nanargmax(likelihood_normalized)
            max_duration.append(duration_val[ind_max])
            max_severity.append(severity_val[ind_max])

    
        out_duration.append(max_duration)
        out_severity.append(max_severity)

## Export the data

In [ ]:
out_duration = pd.DataFrame(out_duration, index=gages.index, columns=[f'D_{t}' for t in T])
out_severity = pd.DataFrame(out_severity, index=gages.index, columns=[f'S_{t}' for t in T])

out_duration = out_duration.round(0)
out_severity = out_severity.round(0)

In [ ]:
temp_df = pd.concat([out_duration, out_severity], axis=1)
out_df = pd.merge(gages, temp_df, left_index=True, right_index=True)

In [ ]:
if EXPORTS:
    
    out_df.to_csv(f'RP_data_{MODEL}_streamgages.csv', index=True)